# **Code Demonstration for Fine Tuning Using LoRA**
This notebook shows how we can fine tune a LLM. In this notebook we fine tune `RoBERTa` on the `IMDb Movie Reviews Dataset`. The `IMDb Movie Reviews Dataset`is commonly used for sentiment analysis and natural language processing (NLP) tasks. It consists of a large collection of movie reviews along with corresponding sentiment labels, indicating whether a review is positive or negative in sentiment

# **Installing and Importing Required Dependencies**
For this demonstartion we us the `datasets` module to import the imdb dataset, `transformers` to import and fine tune `RoBERTa`. `peft` module is used for finetuing

In [ ]:
# !pip install datasets
# !pip install transformers
# !pip install peft
# !pip install evaluate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 18.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 35.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 50.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 100.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 85.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 28.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.18.0
    Uninstalling huggingface-hub-0.18.0:
      Successfully uninstalled huggingface-hub-0.18.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 11.2 MB/s eta 0:

In [ ]:
from datasets import load_dataset, DatasetDict, Dataset

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer)

from peft import PeftModel, PeftConfig, get_peft_model, LoraConfig
import evaluate
import torch
import numpy as np

# **Loading the IMDB dataset**

In [ ]:
from datasets import load_dataset

# Load the IMDb dataset
imdb_dataset = load_dataset("imdb")

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

# **Import Model and Initialize model and tokenizer**

We need to set up a sentiment analysis model using the 'roberta-base' architecture for binary sentiment classification. We define label mappings for 'Negative' and 'Positive' sentiments, then initialize the RoBERTa model for sequence classification with two possible output labels. Furthermore, we need to  create a tokenizer associated with the 'roberta-base' model, ensuring consistent handling of spaces in text by adding a prefix space. If the tokenizer lacks a padding token, it's added as '[PAD]' to facilitate sequence padding during training or inference, and the model's token embeddings are resized accordingly.

In [ ]:
model_checkpoint = 'roberta-base'

# Define label maps for binary sentiment classification
id2label = {0: "Negative", 1: "Positive"}
label2id = {"Negative": 0, "Positive": 1}

# Initialize the model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, num_labels=2, id2label=id2label, label2id=label2id)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

# Add a pad token if not already present
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.out_proj.bias', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# **Tokenize IMDb Dataset for RoBERTa Model Input**
The code defines a tokenize_function responsible for processing text data from the IMDb dataset using the Hugging Face tokenizer. It tokenizes the text field within each example, specifying left-sided truncation and employing the tokenizer to convert the text into numpy arrays. With a maximum token length of 512 (adjustable), the function tokenizes the text and prepares it for input to machine learning models like RoBERTa. This function is then applied to the IMDb dataset using .map() in batches, resulting in the tokenized_dataset variable that stores the tokenized representations of the text data, optimized for RoBERTa model consumption. Adjust the max_length parameter based on specific modeling and sequence length requirements.



In [ ]:
def tokenize_function(examples):
    text = examples["text"]
    tokenizer.truncation_side = "left"
    tokenized_inputs = tokenizer(
        text,
        return_tensors="np",
        truncation=True,
        max_length=512  # Adjust the max_length as needed
    )
    return tokenized_inputs

# Tokenize the IMDb dataset
tokenized_dataset = imdb_dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

`DataCollatorWithPadding` is useful for organizing and preparing data for model training by padding sequences to ensure uniform length, essential for efficient batch processing during model training.

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# **Apply untrained model to text**

In [ ]:
# define list of examples
text_list = ["I love this movie", "I hate this move", "I feel elated after watching this movie"]

print("Untrained model predictions:")
print("----------------------------")
for text in text_list:
    # tokenize text
    inputs = tokenizer.encode(text, return_tensors="pt")
    # compute logits
    logits = model(inputs).logits
    # convert logits to label
    predictions = torch.argmax(logits)

    print(text + " - " + id2label[predictions.tolist()])

Untrained model predictions:
----------------------------
I love this movie - Negative
I hate this move - Negative
I feel elated after watching this movie - Negative


# **Compute accuracy for Prediction**

In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy.compute(predictions=predictions, references=labels)}





# **Setup the LoraConfig**
Now, we need to define the parameters for Low Rank Adaptation.

In [ ]:
peft_config = LoraConfig(task_type="SEQ_CLS",
                        r=4,
                        lora_alpha=32,
                        lora_dropout=0.01,
                        target_modules = ['query'])

In [ ]:

peft_config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type='SEQ_CLS', inference_mode=False, r=4, target_modules=['query'], lora_alpha=32, lora_dropout=0.01, fan_in_fan_out=False, bias='none', modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None)

# **Convert the model into Peft Model**
The `get_peft_model` reduces the number of trainable parameters of the LLM. It uses the above defined parameters to reduce the trainable parameters of the LLM

In [ ]:

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 1,257,988 || all params: 125,313,028 || trainable%: 1.0038764684546606


# **Fine Tuning the Model**
Now we fine tune the model. First, we define the learning rate, number of epoch and batch size. Then we use the trainer function from huggin face to fine tune the model

In [ ]:
# Define your specific training configuration (learning rate, batch size, etc.)
lr = 1e-3
batch_size = 16
num_epochs = 5

training_args = TrainingArguments(
    output_dir="imdb_sentiment_model_checkpoint",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()


You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.204800,0.223865,{'accuracy': 0.93048}
2,0.192300,0.151528,{'accuracy': 0.94672}
3,0.156800,0.146033,{'accuracy': 0.9506}
4,0.125800,0.157026,{'accuracy': 0.95172}
5,0.110500,0.169804,{'accuracy': 0.95208}


Trainer is attempting to log a value of "{'accuracy': 0.93048}" of type <class 'dict'> for key "eval/accuracy" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'accuracy': 0.94672}" of type <class 'dict'> for key "eval/accuracy" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'accuracy': 0.9506}" of type <class 'dict'> for key "eval/accuracy" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'accuracy': 0.95172}" of type <class 'dict'> for key "eval/accuracy" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'accuracy': 0.95208}" of type <class 'dict'> for key "eval/accuracy" as a scala

TrainOutput(global_step=7815, training_loss=0.16066869929747482, metrics={'train_runtime': 12362.6931, 'train_samples_per_second': 10.111, 'train_steps_per_second': 0.632, 'total_flos': 3.271847721139872e+16, 'train_loss': 0.16066869929747482, 'epoch': 5.0})

# **Apply Fine Tuned Model to Text**

In [ ]:
# Move the model to the CPU for inference
model.to('cpu')

print("Trained model predictions:")
print("--------------------------")
text_list = ["I absolutely loved this movie!", "The acting was terrible.", "It was a great film.", "I couldn't stand it.", "Highly recommended!", "It's a waste of time."]

for text in text_list:
    inputs = tokenizer.encode(text, return_tensors="pt").to("cpu")

    with torch.no_grad():  # Disable gradient computation
        logits = model(inputs).logits
    predictions = torch.argmax(logits, dim=1)

    predicted_label = id2label[predictions.item()]
    print(f"{text} - {predicted_label}")


Trained model predictions:
--------------------------
I absolutely loved this movie! - Positive
The acting was terrible. - Negative
It was a great film. - Positive
I couldn't stand it. - Negative
Highly recommended! - Positive
It's a waste of time. - Negative
